In [1]:
from pathlib import Path

# Notebook is inside the "notebooks" folder
PROJECT_ROOT = Path.cwd().parent

# ZIP is inside the "data" folder
zip_path = PROJECT_ROOT / "data" / "TBX11K.zip"

print("Current notebook folder:", Path.cwd())
print("Project folder:", PROJECT_ROOT)
print("ZIP path:", zip_path)
print("ZIP found:", zip_path.exists())

Current notebook folder: C:\Users\A\Desktop\msc data science thesis\notebooks
Project folder: C:\Users\A\Desktop\msc data science thesis
ZIP path: C:\Users\A\Desktop\msc data science thesis\data\TBX11K.zip
ZIP found: True


In [2]:
# Check that the TBX11K ZIP file exists and inspect its folder structure
import os
import zipfile

# Check that the ZIP exists
print("ZIP exists:", os.path.exists(zip_path))

# Read everything inside the ZIP
with zipfile.ZipFile(zip_path, "r") as zip_file:
    items = zip_file.namelist()

print("Total items:", len(items))

# Show all folders found inside the ZIP
folders = sorted({
    item.rstrip("/")
    for item in items
    if item.endswith("/")
})

print("\nFolders inside the ZIP:")
for folder in folders:
    print(folder)

ZIP exists: True
Total items: 13119

Folders inside the ZIP:
TBX11K
TBX11K/annotations
TBX11K/annotations/json
TBX11K/annotations/xml
TBX11K/code
TBX11K/imgs
TBX11K/imgs/extra
TBX11K/imgs/extra/da+db
TBX11K/imgs/extra/da+db/train
TBX11K/imgs/extra/da+db/val
TBX11K/imgs/extra/mc+shenzhen
TBX11K/imgs/extra/mc+shenzhen/train
TBX11K/imgs/extra/mc+shenzhen/val
TBX11K/imgs/health
TBX11K/imgs/sick
TBX11K/imgs/tb
TBX11K/imgs/test
TBX11K/lists


In [3]:
# Count files inside each imgs subfolder
from collections import Counter

counts = Counter()

for item in items:
    if item.endswith("/"):   # Ignore folders
        continue

    parts = item.split("/")

    # Count files inside TBX11K/imgs/
    if len(parts) >= 4 and parts[1] == "imgs":
        counts[parts[2]] += 1

print(counts)

Counter({'sick': 3800, 'health': 3800, 'test': 3302, 'tb': 800, 'extra': 576})


In [4]:
#inspecting the list folder
list_files = [
    item for item in items
    if item.startswith("TBX11K/lists/") and not item.endswith("/")
]

print("Files inside lists folder:")
for file in list_files:
    print(file)

Files inside lists folder:
TBX11K/lists/all_trainval.txt
TBX11K/lists/TBX11K_trainval.txt
TBX11K/lists/all_val.txt
TBX11K/lists/all_train.txt
TBX11K/lists/all_test.txt
TBX11K/lists/TBX11K_train.txt
TBX11K/lists/TBX11K_val.txt


In [5]:
# Read the official training and validation image paths and remove blank lines
with zipfile.ZipFile(zip_path, "r") as zip_file:

    train_files = zip_file.read(
        "TBX11K/lists/TBX11K_train.txt"
    ).decode("utf-8").splitlines()

    val_files = zip_file.read(
        "TBX11K/lists/TBX11K_val.txt"
    ).decode("utf-8").splitlines()

# Remove empty lines
train_files = [x.strip() for x in train_files if x.strip()]
val_files = [x.strip() for x in val_files if x.strip()]

print("Training image paths:", len(train_files))
print("Validation image paths:", len(val_files))

print("\nFirst 5 training paths:")
for path in train_files[:5]:
    print(path)

print("\nFirst 5 validation paths:")
for path in val_files[:5]:
    print(path)

Training image paths: 6600
Validation image paths: 1800

First 5 training paths:
tb/tb0005.png
tb/tb0007.png
tb/tb0012.png
tb/tb0017.png
tb/tb0018.png

First 5 validation paths:
tb/tb0003.png
tb/tb0004.png
tb/tb0006.png
tb/tb0009.png
tb/tb0014.png


In [6]:
# Check that train and validation paths exist and use valid class folders

zip_items = set(items)
allowed_folders = {"health", "sick", "tb"}

def check_paths(file_paths):
    missing = []
    invalid_folders = []

    for path in file_paths:
        folder = path.split("/")[0]
        full_path = f"TBX11K/imgs/{path}"

        if folder not in allowed_folders:
            invalid_folders.append(path)

        if full_path not in zip_items:
            missing.append(path)

    return missing, invalid_folders


missing_train, invalid_train = check_paths(train_files)
missing_val, invalid_val = check_paths(val_files)

print("Missing training images:", len(missing_train))
print("Missing validation images:", len(missing_val))
print("Invalid training folder paths:", len(invalid_train))
print("Invalid validation folder paths:", len(invalid_val))

Missing training images: 0
Missing validation images: 0
Invalid training folder paths: 0
Invalid validation folder paths: 0


In [7]:
# Count health, sick and TB image paths in training and validation

from collections import Counter

train_class_counts = Counter(
    path.split("/")[0] for path in train_files
)

val_class_counts = Counter(
    path.split("/")[0] for path in val_files
)

print("Training class counts:", train_class_counts)
print("Validation class counts:", val_class_counts)

Training class counts: Counter({'health': 3000, 'sick': 3000, 'tb': 600})
Validation class counts: Counter({'health': 800, 'sick': 800, 'tb': 200})


In [8]:
# Check duplicate paths and train-validation overlap
train_duplicates = len(train_files) - len(set(train_files))
val_duplicates = len(val_files) - len(set(val_files))
train_val_overlap = set(train_files) & set(val_files)

print("Duplicate training paths:", train_duplicates)
print("Duplicate validation paths:", val_duplicates)
print("Train-validation overlap:", len(train_val_overlap))

Duplicate training paths: 0
Duplicate validation paths: 0
Train-validation overlap: 0


In [9]:
# Check for corrupted or unreadable training and validation images

from io import BytesIO
from PIL import Image

corrupted_images = []
all_trainval = sorted(set(train_files + val_files))

with zipfile.ZipFile(zip_path, "r") as zip_file:
    for path in all_trainval:
        full_path = f"TBX11K/imgs/{path}"

        try:
            with Image.open(BytesIO(zip_file.read(full_path))) as image:
                image.load()
        except Exception as error:
            corrupted_images.append((path, str(error)))

print("Images checked:", len(all_trainval))
print("Corrupted or unreadable images:", len(corrupted_images))

Images checked: 8400
Corrupted or unreadable images: 0


In [10]:
# Check image dimensions, colour mode and file format

from collections import Counter
from io import BytesIO
from PIL import Image
import zipfile

image_sizes = Counter()
image_modes = Counter()
image_formats = Counter()

all_trainval = sorted(set(train_files + val_files))

with zipfile.ZipFile(zip_path, "r") as zip_file:
    for path in all_trainval:
        full_path = f"TBX11K/imgs/{path}"

        with Image.open(BytesIO(zip_file.read(full_path))) as image:
            image.load()

            image_sizes[image.size] += 1
            image_modes[image.mode] += 1
            image_formats[image.format] += 1

print("Image dimensions:", image_sizes)
print("Image colour modes:", image_modes)
print("Image formats:", image_formats)

Image dimensions: Counter({(512, 512): 8400})
Image colour modes: Counter({'RGB': 8400})
Image formats: Counter({'PNG': 8400})


In [11]:
# Count XML and JSON annotation files inside the ZIP

from collections import Counter
import os

annotation_counts = Counter()

for item in items:
    if item.startswith("TBX11K/annotations/") and not item.endswith("/"):
        extension = os.path.splitext(item)[1].lower()
        annotation_counts[extension] += 1

print("Annotation file counts:", annotation_counts)

Annotation file counts: Counter({'.xml': 800, '.json': 10})
